# Ball heading through short ATD gaps

## Note: Turns out this is complete bullshit and this prediction method is basically unusable


Reuse the detection-interval logic from [`01_ball_detection_timeline.ipynb`](01_ball_detection_timeline.ipynb):
find missing-ball gaps shorter than `MISSING_BALL_INTERVAL_LEN_FR`, estimate the ball's
**velocity just before** each gap, coast with **constant velocity** across the gap, and measure
how far (in metres) the predicted reappearance is from the ATD's next observed position.

Pitch size comes from `games.py` (`105.3 × 68.0` m). ATD coordinates are normalised 0–1.

## Setup and ball presence

Same raw-ATD streaming pattern as the timeline notebook: `frame_id:players:ball`, missing ball
starts with `N`. Here we also keep **normalised** `(x, y)` for every present frame so we can
build a velocity vector.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('../..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import games

FIELD_SEP = ':'
MISSING = 'N'  # first character of "NaN"

paths = games.asset_paths(source='tactical')
n_frames = games.GAMES[games.GAME]['n_video_frames']
fps = games.GAMES[games.GAME]['fps']
pitch_length, pitch_width = games.GAMES[games.GAME]['pitch']  # metres

print('game:     ', games.GAME)
print('frames:   ', n_frames)
print('fps:      ', fps)
print('pitch:    ', pitch_length, 'x', pitch_width, 'm')
print('atd_raw:  ', paths['atd_raw'])
print('exists:   ', os.path.isfile(paths['atd_raw']))

In [ ]:
def load_ball_xy(raw_path, n_frames):
    """Per-frame ball (x, y) in normalised 0–1 pitch coords; NaN where missing.

    Indexing matches the submission / tactical video (0 .. n_frames-1).
    Raw ball field is `x,y,speed` (or `NaN,...`).
    """
    xy = np.full((n_frames, 2), np.nan, dtype=np.float64)
    with open(raw_path) as handle:
        for line in handle:
            parts = line.split(FIELD_SEP)
            if len(parts) != 3:
                continue
            ball = parts[2]
            if ball[0] == MISSING:
                continue
            frame_id = int(parts[0])
            comma = ball.index(',')
            x = float(ball[:comma])
            y = float(ball[comma + 1:ball.index(',', comma + 1)])
            xy[frame_id - 1] = (x, y)
    return xy


ball_xy_norm = load_ball_xy(paths['atd_raw'], n_frames)
ball_present_full = np.isfinite(ball_xy_norm[:, 0])
# Metres on the real pitch (asymmetric: length ≠ width).
ball_xy_m = ball_xy_norm.copy()
ball_xy_m[:, 0] *= pitch_length
ball_xy_m[:, 1] *= pitch_width

n_present_full = int(ball_present_full.sum())
print('full video — ball present: %d / %d frames (%.1f%%)' % (
    n_present_full, n_frames, 100 * n_present_full / n_frames))

## Scope and short missing intervals

Same `GRADED_ONLY` / half-block cutting as the timeline notebook so gaps never span halftime.
Keep only **absent** runs with `1 ≤ n_frames ≤ MISSING_BALL_INTERVAL_LEN_FR`.

In [ ]:
# --- knobs ---
GRADED_ONLY = False
MISSING_BALL_INTERVAL_LEN_FR = 50   # keep gaps of this many frames or fewer
VEL_LOOKBACK_FR = 100                # frames before the gap used to estimate velocity

half_blocks = games.half_video_blocks(games.GAME)
SCOPE_LABEL = 'graded halves' if GRADED_ONLY else 'full match'
analysis_blocks = half_blocks if GRADED_ONLY else [(None, 0, n_frames - 1)]


def runs(mask):
    """Yield (start, length, value) for each contiguous run in a 1-d boolean array."""
    if len(mask) == 0:
        return
    changes = np.flatnonzero(mask[1:] != mask[:-1]) + 1
    bounds = np.concatenate(([0], changes, [len(mask)]))
    for i in range(len(bounds) - 1):
        start, end = int(bounds[i]), int(bounds[i + 1])
        yield start, end - start, bool(mask[start])


def intervals_in_blocks(ball_mask, blocks):
    """Runs of present/absent inside each analysis block (video-frame indexing)."""
    out = []
    for half, first, last in blocks:
        for s, L, p in runs(ball_mask[first:last + 1]):
            out.append((first + s, L, p, half))
    return out


intervals = intervals_in_blocks(ball_present_full, analysis_blocks)
absent_all = [(s, L, half) for s, L, p, half in intervals if not p]

short_gaps = [(s, L, half) for s, L, half in absent_all
              if 1 <= L <= MISSING_BALL_INTERVAL_LEN_FR]

print('scope:              %s' % SCOPE_LABEL)
print('absent intervals:   %d' % len(absent_all))
print('short gaps (≤ %d):  %d  (%.1f%% of absent)' % (
    MISSING_BALL_INTERVAL_LEN_FR, len(short_gaps),
    100 * len(short_gaps) / max(len(absent_all), 1)))
print('median short gap:   %d frames (%.2f s)' % (
    np.median([L for _, L, _ in short_gaps]),
    np.median([L for _, L, _ in short_gaps]) / fps))

## Constant-velocity coast across each short gap

For a missing run starting at frame `s` with length `L`:

| | frame |
|---|---|
| last seen | `s - 1` |
| missing | `s … s + L - 1` |
| reappears | `s + L` |

Velocity (m/s) is the mean frame-to-frame displacement over up to `VEL_LOOKBACK_FR` consecutive
present frames ending at the last seen frame, times `fps`. Predicted reappearance:

```
p_pred = p_last + v * (L + 1) / fps
```

Error is Euclidean distance in metres between `p_pred` and the ATD position at `s + L`.
Gaps without enough history before, or without a present frame after, are skipped.

In [ ]:
def estimate_velocity_mps(xy_m, last_frame, lookback, fps):
    """Mean (vx, vy) in m/s from consecutive present samples ending at last_frame.

    Needs at least two present frames in [last_frame - lookback + 1, last_frame]
    with no holes between the earliest and latest used sample (so the gap itself is
    not crossed). Returns None if the window is too thin.
    """
    lo = max(0, last_frame - lookback + 1)
    window = xy_m[lo:last_frame + 1]
    present = np.isfinite(window[:, 0])
    if present.sum() < 2 or not present[-1]:
        return None
    # Restrict to the trailing contiguous present run (ending at last_frame).
    # If the lookback itself contains a hole, only use frames after the last hole.
    if not present.all():
        # index of last False in present, relative to window
        last_miss = np.flatnonzero(~present)[-1]
        window = window[last_miss + 1:]
        if len(window) < 2:
            return None
    dxy = np.diff(window, axis=0)  # metres per frame
    return dxy.mean(axis=0) * fps  # m/s


def predict_short_gaps(xy_m, gaps, lookback, fps):
    """One row per evaluable short gap with predicted vs observed reappearance."""
    rows = []
    n = len(xy_m)
    for s, L, half in gaps:
        last_seen = s - 1
        reappear = s + L
        if last_seen < 0 or reappear >= n:
            continue
        if not np.isfinite(xy_m[last_seen, 0]) or not np.isfinite(xy_m[reappear, 0]):
            continue

        vel = estimate_velocity_mps(xy_m, last_seen, lookback, fps)
        if vel is None:
            continue

        dt = (reappear - last_seen) / fps  # = (L + 1) / fps
        p_last = xy_m[last_seen]
        p_obs = xy_m[reappear]
        p_pred = p_last + vel * dt
        err_m = float(np.hypot(*(p_pred - p_obs)))
        speed = float(np.hypot(*vel))

        rows.append({
            'start_frame': s,
            'end_frame': s + L - 1,
            'n_frames': L,
            'duration_s': L / fps,
            'half': half,
            'last_seen': last_seen,
            'reappear': reappear,
            'vx_mps': float(vel[0]),
            'vy_mps': float(vel[1]),
            'speed_mps': speed,
            'x_last': float(p_last[0]),
            'y_last': float(p_last[1]),
            'x_pred': float(p_pred[0]),
            'y_pred': float(p_pred[1]),
            'x_obs': float(p_obs[0]),
            'y_obs': float(p_obs[1]),
            'error_m': err_m,
        })
    return pd.DataFrame(rows)


pred_df = predict_short_gaps(ball_xy_m, short_gaps, VEL_LOOKBACK_FR, fps)

print('short gaps considered:     %d' % len(short_gaps))
print('evaluable (with velocity): %d' % len(pred_df))
if len(pred_df):
    print()
    print('endpoint error (m):')
    print(pred_df['error_m'].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95]).to_string())
    print()
    print('share error ≤ 1 m:  %.1f%%' % (100 * (pred_df['error_m'] <= 1).mean()))
    print('share error ≤ 3 m:  %.1f%%' % (100 * (pred_df['error_m'] <= 3).mean()))
    print('share error ≤ 5 m:  %.1f%%' % (100 * (pred_df['error_m'] <= 5).mean()))
pred_df.head()

## How good is the coast?

Three views of the same endpoint errors:

1. **Histogram + CDF** — overall accuracy of constant-velocity prediction.
2. **Error vs gap length** — whether longer coasts drift more (as expected under constant `v`).
3. **Error vs pre-gap speed** — whether fast balls (passes, clearances) are harder.

In [ ]:
assert len(pred_df), 'No evaluable short gaps — check ATD path / knobs.'

err = pred_df['error_m'].to_numpy()
COLOR = '#d73027'

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

# --- 1. Histogram + CDF ---
ax = axes[0]
bins = np.linspace(0, min(np.percentile(err, 99), max(err.max(), 1e-6)), 40)
ax.hist(err, bins=bins, color=COLOR, edgecolor='white', linewidth=0.4, alpha=0.85)
ax.axvline(np.median(err), color='black', ls='--', lw=1.2,
           label='median %.2f m' % np.median(err))
ax.axvline(np.mean(err), color='black', ls=':', lw=1.2,
           label='mean %.2f m' % np.mean(err))
ax.set_xlabel('endpoint error (m)')
ax.set_ylabel('number of gaps')
ax.set_title('Error distribution')
ax.legend(fontsize=8, framealpha=0.9)

ax_cdf = ax.twinx()
sorted_err = np.sort(err)
ax_cdf.plot(sorted_err, np.linspace(0, 1, len(sorted_err)),
            color='#4575b4', lw=1.5)
ax_cdf.set_ylabel('CDF', color='#4575b4')
ax_cdf.set_ylim(0, 1)
ax_cdf.tick_params(axis='y', labelcolor='#4575b4')

# --- 2. Error vs gap length ---
ax = axes[1]
ax.scatter(pred_df['n_frames'], pred_df['error_m'],
           s=12, alpha=0.35, c=COLOR, linewidths=0, rasterized=True)
# Binned median for trend.
max_L = int(pred_df['n_frames'].max())
bins_L = np.arange(1, max_L + 2)
centers, medians = [], []
for lo, hi in zip(bins_L[:-1], bins_L[1:]):
    sel = pred_df['n_frames'] == lo
    if sel.any():
        centers.append(lo)
        medians.append(pred_df.loc[sel, 'error_m'].median())
ax.plot(centers, medians, color='black', lw=1.8, label='median per length')
ax.set_xlabel('gap length (frames)')
ax.set_ylabel('endpoint error (m)')
ax.set_title('Error vs gap length')
ax.set_xlim(0.5, MISSING_BALL_INTERVAL_LEN_FR + 0.5)
ax.legend(fontsize=8, framealpha=0.9)

ax_sec = ax.twiny()
ax_sec.set_xlim(ax.get_xlim()[0] / fps, ax.get_xlim()[1] / fps)
ax_sec.set_xlabel('gap length (seconds)')

# --- 3. Error vs pre-gap speed ---
ax = axes[2]
ax.scatter(pred_df['speed_mps'], pred_df['error_m'],
           s=12, alpha=0.35, c='#4575b4', linewidths=0, rasterized=True)
ax.set_xlabel('pre-gap speed (m/s)')
ax.set_ylabel('endpoint error (m)')
ax.set_title('Error vs pre-gap speed')

fig.suptitle(
    'Constant-velocity ball coast across short ATD gaps\n'
    '(%s, gap ≤ %d frames, lookback %d frames, n = %d)' % (
        SCOPE_LABEL, MISSING_BALL_INTERVAL_LEN_FR, VEL_LOOKBACK_FR, len(pred_df)),
    y=1.05)
plt.tight_layout()
plt.show()

## Residual vectors on the pitch

Each arrow goes from the **predicted** reappearance to the **observed** one (metres on the
pitch). Short arrows mean the coast landed close; systematic fans would hint at bias
(e.g. always overshooting along the length).

In [ ]:
fig, ax = plt.subplots(figsize=(12, 12 * pitch_width / pitch_length))

# Pitch outline (metres, origin bottom-left in ATD convention).
ax.add_patch(plt.Rectangle((0, 0), pitch_length, pitch_width,
                           fill=False, ec='0.3', lw=1.2))
ax.axvline(pitch_length / 2, color='0.5', ls='--', lw=0.8)

dx = pred_df['x_obs'] - pred_df['x_pred']
dy = pred_df['y_obs'] - pred_df['y_pred']
# Subsample if dense so the quiver stays readable.
rng = np.random.default_rng(0)
idx = np.arange(len(pred_df))
if len(idx) > 800:
    idx = rng.choice(idx, size=800, replace=False)

q = ax.quiver(
    pred_df['x_pred'].iloc[idx], pred_df['y_pred'].iloc[idx],
    dx.iloc[idx], dy.iloc[idx],
    pred_df['error_m'].iloc[idx],
    angles='xy', scale_units='xy', scale=1,
    cmap='YlOrRd', width=0.0025, alpha=0.7,
)
cb = fig.colorbar(q, ax=ax, fraction=0.03, pad=0.02)
cb.set_label('endpoint error (m)')

ax.set_xlim(-2, pitch_length + 2)
ax.set_ylim(-2, pitch_width + 2)
ax.set_aspect('equal')
ax.set_xlabel('x (m)')
ax.set_ylabel('y (m)')
ax.set_title('Residual: observed − predicted reappearance\n'
             '(arrow from prediction to ATD observation)')
plt.tight_layout()
plt.show()

print('mean residual (obs − pred):  dx = %+.2f m,  dy = %+.2f m' % (
    dx.mean(), dy.mean()))
print('RMS residual:                %.2f m' % np.sqrt((dx**2 + dy**2).mean()))

## Worst coasts (optional peek)

Largest endpoint misses — useful to spot tracker jumps at reappearance or velocity estimated
across a bounce / deflection.

In [ ]:
cols = ['start_frame', 'n_frames', 'duration_s', 'speed_mps',
        'vx_mps', 'vy_mps', 'error_m', 'half']
pred_df.sort_values('error_m', ascending=False)[cols].head(15)